## 데이터 기반 ai 모델링 코드

In [ ]:
# ==============================================================================
# 🚀 [Ultimate] CSRNet - 시장(전체) + 엑스포(15,000장) 맞춤형 & 과적합 완벽 방어
# ==============================================================================

import os
import glob
import json
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
import scipy.ndimage as ndimage
import scipy.spatial as spatial
import shutil

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# ---------------------------------------------------------
# 0. 구글 드라이브 마운트 및 데이터 압축 해제
# ---------------------------------------------------------
DRIVE_SAVE_PATH = '/content/drive/MyDrive/csrnet_checkpoints'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ 구글 드라이브 마운트 완료!")

    if not os.path.exists(DRIVE_SAVE_PATH):
        os.makedirs(DRIVE_SAVE_PATH)
        print(f"📁 구글 드라이브에 저장 폴더 생성 완료: {DRIVE_SAVE_PATH}")

    def extract_dataset(zip_name, target_folder):
        zip_path = f"/content/drive/MyDrive/{zip_name}"
        if not os.path.exists(f"/content/{target_folder}"):
            if not os.path.exists(zip_path):
                print(f"⚠️ 경고: '{zip_path}' 파일이 드라이브에 없습니다.")
                return
            print(f"⏳ '{zip_name}' 압축 해제 중...")
            os.system(f'unzip -q -o "{zip_path}" -d /content/')
            print(f"✅ '{zip_name}' 압축 해제 완료!")

    extract_dataset('CrowdData.zip', 'CrowdData')
    extract_dataset('CrowdData_Expo.zip', 'CrowdData_Expo')

except ImportError:
    print("⚠️ 구글 코랩 환경이 아닙니다.")

# ---------------------------------------------------------
# 1. 하이퍼파라미터 설정
# ---------------------------------------------------------
BATCH_SIZE = 2
EPOCHS = 15
LR = 1e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CROP_SIZE = 512

# ---------------------------------------------------------
# 2. 맞춤형 데이터셋 로더 (시장 100% + 엑스포 15,000장)
# ---------------------------------------------------------
class DualCrowdDataset(Dataset):
    def __init__(self):
        print("\n🔍 두 데이터셋 탐색 및 [엑스포 1.5만장 제한] 적용 중...")

        market_jsons = list(Path('/content/CrowdData').rglob('*.json'))
        expo_jsons = list(Path('/content/CrowdData_Expo').rglob('*.json'))

        print(f"   - 원본 시장 데이터: {len(market_jsons)}개")
        print(f"   - 원본 엑스포 데이터: {len(expo_jsons)}개")

        # 💡 [핵심 변경] 시장 데이터는 건드리지 않고, 엑스포만 15,000장으로 대폭 축소!
        TARGET_EXPO_COUNT = 15000
        random.seed(42)

        if len(expo_jsons) > TARGET_EXPO_COUNT:
            expo_jsons = random.sample(expo_jsons, TARGET_EXPO_COUNT)

        print(f"   ➡ 데이터 확정: 시장 {len(market_jsons)}개 (100%) / 엑스포 {len(expo_jsons)}개 (축소) 사용")

        all_jsons = market_jsons + expo_jsons

        # 이미지는 전체 탐색
        all_imgs = list(Path('/content/CrowdData').rglob('*.jpg')) + list(Path('/content/CrowdData_Expo').rglob('*.jpg'))
        img_dict = {f.stem: str(f) for f in all_imgs}

        self.matched_pairs = []
        for json_path in all_jsons:
            stem = json_path.stem
            if stem in img_dict:
                self.matched_pairs.append((img_dict[stem], str(json_path)))

        if len(self.matched_pairs) == 0:
            raise RuntimeError("❌ 훈련할 데이터를 단 1장도 찾지 못했습니다!")

        print(f"✅ 최종 훈련에 사용될 [이미지-라벨] 짝: 총 {len(self.matched_pairs)}쌍")

        random.seed(42)
        random.shuffle(self.matched_pairs)

        # 🌟 [과적합 방어 1] Color Jitter
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.matched_pairs)

    def __getitem__(self, idx):
        img_path, json_path = self.matched_pairs[idx]

        # 1. 이미지 로드
        img = cv2.imread(img_path)
        if img is None:
            img = np.zeros((1080, 1920, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = img.shape[:2]

        # 2. JSON 파싱
        try:
            with open(json_path, 'r', encoding='utf-8-sig') as f:
                data = json.load(f)
        except Exception:
            data = {}

        pts = []
        if 'image' in data and 'crowdinfo' in data['image'] and 'objects' in data['image']['crowdinfo']:
            for obj in data['image']['crowdinfo']['objects']:
                pt = obj.get('directionindex', [])
                if pt and len(pt) >= 2:
                    pts.append([float(pt[0]), float(pt[1])])

        # 🌟 [과적합 방어 2] 랜덤 크롭
        if orig_h > CROP_SIZE and orig_w > CROP_SIZE:
            dy = random.randint(0, orig_h - CROP_SIZE)
            dx = random.randint(0, orig_w - CROP_SIZE)
            img = img[dy:dy+CROP_SIZE, dx:dx+CROP_SIZE]

            new_pts = []
            for pt in pts:
                nx, ny = pt[0] - dx, pt[1] - dy
                if 0 <= nx < CROP_SIZE and 0 <= ny < CROP_SIZE:
                    new_pts.append([nx, ny])
            pts = new_pts
        else:
            img = cv2.resize(img, (CROP_SIZE, CROP_SIZE))
            scale_x = CROP_SIZE / orig_w
            scale_y = CROP_SIZE / orig_h
            pts = [[p[0]*scale_x, p[1]*scale_y] for p in pts]

        # 🌟 [과적합 방어 3] 좌우 반전
        is_flipped = False
        if random.random() > 0.5:
            img = cv2.flip(img, 1)
            is_flipped = True

        out_w, out_h = CROP_SIZE // 8, CROP_SIZE // 8
        density_map = np.zeros((out_h, out_w), dtype=np.float32)

        # 🌟 [핵심] 기하학적 적응형 가우시안
        if len(pts) > 0:
            pts_array = np.array(pts) / 8.0

            if len(pts_array) > 1:
                tree = spatial.KDTree(pts_array)
                distances, _ = tree.query(pts_array, k=2)
                distances = distances[:, 1]
            else:
                distances = np.array([15.0])

            for i, pt in enumerate(pts_array):
                pt_x, pt_y = int(pt[0]), int(pt[1])

                if is_flipped:
                    pt_x = out_w - 1 - pt_x

                if 0 <= pt_x < out_w and 0 <= pt_y < out_h:
                    sigma = max(1.0, min(7.0, distances[i] * 0.5))
                    temp_map = np.zeros((out_h, out_w), dtype=np.float32)
                    temp_map[pt_y, pt_x] = 1.0
                    density_map += ndimage.gaussian_filter(temp_map, sigma=sigma, mode='constant')

        img_tensor = self.transform(img)
        density_tensor = torch.from_numpy(density_map.copy())

        return img_tensor, density_tensor

# ---------------------------------------------------------
# 3. CSRNet 모델 정의 (뼈대)
# ---------------------------------------------------------
class CSRNet(nn.Module):
    def __init__(self):
        super(CSRNet, self).__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        features = list(vgg.features.children())
        self.frontend = nn.Sequential(*features[0:23])
        self.backend = nn.Sequential(
            nn.Conv2d(512, 512, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(512, 256, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1)
        )

    def forward(self, x):
        x = self.frontend(x)
        x = self.backend(x)
        return x

# ---------------------------------------------------------
# 4. 훈련 루프
# ---------------------------------------------------------
def train_model():
    print(f"\n🚀 최고의 맞춤형 모델 훈련 준비 중... (디바이스: {DEVICE})")

    dataset = DualCrowdDataset()
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    model = CSRNet().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss(reduction='sum')

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        epoch_mae = 0.0

        pbar = tqdm(dataloader, desc=f"Epoch [{epoch+1}/{EPOCHS}]")
        try:
            for imgs, target_maps in pbar:
                imgs = imgs.to(DEVICE)
                target_maps = target_maps.unsqueeze(1).to(DEVICE)

                pred_maps = model(imgs)

                loss = criterion(pred_maps, target_maps)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                pred_count = pred_maps.sum().item()
                true_count = target_maps.sum().item()
                mae = abs(pred_count - true_count)

                epoch_loss += loss.item()
                epoch_mae += mae

                pbar.set_postfix({'Loss': loss.item(), 'MAE (Error)': mae})

            print(f"🔥 Epoch [{epoch+1}/{EPOCHS}] 완료! - 평균 오차(MAE): {epoch_mae/len(dataloader):.2f}명")

            save_name = f'csrnet_ultimate_epoch_{epoch+1}.pth'
            torch.save(model.state_dict(), save_name)
            print(f"💾 중간 저장 완료: {save_name}")

            if os.path.exists('/content/drive/MyDrive'):
                shutil.copy(save_name, os.path.join(DRIVE_SAVE_PATH, save_name))
                print(f"☁️ 구글 드라이브 영구 저장 완료: {save_name}")

        except KeyboardInterrupt:
            print("\n🚨 사용자가 훈련을 강제 종료했습니다!")
            interrupted_name = 'csrnet_ultimate_interrupted.pth'
            torch.save(model.state_dict(), interrupted_name)

            if os.path.exists('/content/drive/MyDrive'):
                shutil.copy(interrupted_name, os.path.join(DRIVE_SAVE_PATH, interrupted_name))
            return

    best_name = 'csrnet_ultimate_best.pth'
    torch.save(model.state_dict(), best_name)
    print(f"✅ 모든 통합 훈련 완료! 모델이 '{best_name}'로 저장되었습니다.")

    if os.path.exists('/content/drive/MyDrive'):
        shutil.copy(best_name, os.path.join(DRIVE_SAVE_PATH, best_name))

if __name__ == '__main__':
    train_model()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 구글 드라이브 마운트 완료!

🚀 최고의 맞춤형 모델 훈련 준비 중... (디바이스: cuda)

🔍 두 데이터셋 탐색 및 [엑스포 1.5만장 제한] 적용 중...
   - 원본 시장 데이터: 31590개
   - 원본 엑스포 데이터: 63847개
   ➡ 데이터 확정: 시장 31590개 (100%) / 엑스포 15000개 (축소) 사용
✅ 최종 훈련에 사용될 [이미지-라벨] 짝: 총 46590쌍


Epoch [1/15]: 100%|██████████| 23295/23295 [32:21<00:00, 12.00it/s, Loss=0.000507, MAE (Error)=0.822]


🔥 Epoch [1/15] 완료! - 평균 오차(MAE): 4.40명
💾 중간 저장 완료: csrnet_ultimate_epoch_1.pth
☁️ 구글 드라이브 영구 저장 완료: csrnet_ultimate_epoch_1.pth


Epoch [2/15]: 100%|██████████| 23295/23295 [31:03<00:00, 12.50it/s, Loss=0.0117, MAE (Error)=0.792]


🔥 Epoch [2/15] 완료! - 평균 오차(MAE): 2.64명
💾 중간 저장 완료: csrnet_ultimate_epoch_2.pth
☁️ 구글 드라이브 영구 저장 완료: csrnet_ultimate_epoch_2.pth


...

🔥 Epoch [9/15] 완료! - 평균 오차(MAE): 1.82명
💾 중간 저장 완료: csrnet_ultimate_epoch_9.pth
☁️ 구글 드라이브 영구 저장 완료: csrnet_ultimate_epoch_9.pth


Epoch [10/15]: 100%|██████████| 23295/23295 [29:47<00:00, 13.03it/s, Loss=0.002, MAE (Error)=0.116]


🔥 Epoch [10/15] 완료! - 평균 오차(MAE): 1.77명
💾 중간 저장 완료: csrnet_ultimate_epoch_10.pth
☁️ 구글 드라이브 영구 저장 완료: csrnet_ultimate_epoch_10.pth


Epoch [11/15]:  22%|██▏       | 5197/23295 [06:40<23:13, 12.99it/s, Loss=0.0545, MAE (Error)=0.827]



🚨 사용자가 훈련을 강제 종료했습니다!


모델링 파일

## 모델링 csrnet_ultimate_epoch_8.pth파일로 동영상 분석하는 코드

In [ ]:
# ==============================================================================
# 🎬 CSRNet 동영상 밀집도 분석 (흐릿함 제거 + 선명도 완벽 복원 버젼)
# ==============================================================================

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ 구글 드라이브 마운트 완료!")
except ImportError:
    print("⚠️ 구글 코랩 환경이 아닙니다.")

import cv2
import torch
import torch.nn as nn
import numpy as np
import os
import pandas as pd
import shutil
from torchvision import transforms, models
from tqdm.notebook import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 🌟 에포크 8 유지
MODEL_PATH = '/content/drive/MyDrive/csrnet_checkpoints/csrnet_ultimate_epoch_8.pth'
VIDEO_PATH = '/content/drive/MyDrive/cctv_EXCO_test_output.mp4'

# 안전 저장 경로
LOCAL_VIDEO_PATH = '/content/temp_result.mp4'
LOCAL_CSV_PATH = '/content/temp_result.csv'
DRIVE_VIDEO_PATH = '/content/drive/MyDrive/cctv_EXCO_test_output_최종분석결과.mp4'
DRIVE_CSV_PATH = '/content/drive/MyDrive/E05_032_최종인원기록.csv'

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CSRNet(nn.Module):
    def __init__(self):
        super(CSRNet, self).__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        features = list(vgg.features.children())
        self.frontend = nn.Sequential(*features[0:23])
        self.backend = nn.Sequential(
            nn.Conv2d(512, 512, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(512, 256, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1)
        )

    def forward(self, x):
        x = self.frontend(x)
        x = self.backend(x)
        return x

def analyze_video():
    model = CSRNet().to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(LOCAL_VIDEO_PATH, fourcc, fps, (orig_w, orig_h))

    time_list = []
    count_list = []

    pbar = tqdm(total=total_frames, desc="🎬 영상 분석 중")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # 이태원 영상이므로 False 그대로 진행 (원본 해상도로 분석)
        USE_RESIZE = False

        if USE_RESIZE:
            INFER_W, INFER_H = 960, 540
            frame_resized = cv2.resize(frame, (INFER_W, INFER_H))
            img_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
        else:
            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        img_tensor = transform(img_rgb).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            pred_map = model(img_tensor)
            pred_map = torch.clamp(pred_map, min=0)
            pred_count = pred_map.sum().item()

        density_map = pred_map.squeeze().cpu().numpy()
        density_map_resized = cv2.resize(density_map, (orig_w, orig_h))

        # 🌟 [문제 원인 삭제 완료]
        # 화면을 뿌옇게 만들던 GaussianBlur 코드를 완전히 날려버렸습니다!
        # 이제 다시 쨍하고 뾰족한 히트맵이 나올 것입니다.

        max_val = density_map_resized.max()
        if max_val > 0:
            density_map_norm = (density_map_resized / max_val * 255).astype(np.uint8)
        else:
            density_map_norm = np.zeros_like(density_map_resized, dtype=np.uint8)

        heatmap = cv2.applyColorMap(density_map_norm, cv2.COLORMAP_JET)

        # 🌟 [수정 2] 원본 영상 50% + 히트맵 50% (기존 6:4 비율에서 변경)
        # 히트맵의 강도를 조금 더 높여서, 예전처럼 색깔이 선명하게 눈에 확 들어오게 만들었습니다.
        result_frame = cv2.addWeighted(frame, 0.5, heatmap, 0.5, 0)

        # 텍스트 오버레이
        text = f"Estimated Crowd: {int(np.round(pred_count))} people"
        cv2.putText(result_frame, text, (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3, cv2.LINE_AA)

        out.write(result_frame)
        pbar.update(1)

        time_list.append(cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0)
        count_list.append(int(np.round(pred_count)))

    cap.release()
    out.release()
    pbar.close()

    df = pd.DataFrame({
        'Time (sec)': time_list,
        'Crowd Count': count_list
    })
    df.to_csv(LOCAL_CSV_PATH, index=False)

    print("\n⏳ 구글 드라이브로 최종 파일을 저장하고 있습니다...")
    try:
        shutil.copy(LOCAL_VIDEO_PATH, DRIVE_VIDEO_PATH)
        shutil.copy(LOCAL_CSV_PATH, DRIVE_CSV_PATH)
        print(f"✅ 구글 드라이브 안전 저장 완료!")
        print(f" 🎥 영상: {DRIVE_VIDEO_PATH}")
        print(f" 📊 데이터: {DRIVE_CSV_PATH}")
    except Exception as e:
        print(f"⚠️ 드라이브 저장 실패! 오류: {e}")

if __name__ == '__main__':
    analyze_video()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 구글 드라이브 마운트 완료!


🎬 영상 분석 중:   0%|          | 0/489 [00:00<?, ?it/s]


⏳ 구글 드라이브로 최종 파일을 저장하고 있습니다...
✅ 구글 드라이브 안전 저장 완료!
 🎥 영상: /content/drive/MyDrive/cctv_EXCO_test_output_최종분석결과.mp4
 📊 데이터: /content/drive/MyDrive/E05_032_최종인원기록.csv
